In [ ]:
# 本 Python 3 环境预装了多种实用的数据分析库
# 它由 kaggle/python Docker 镜像定义：https://github.com/kaggle/docker-python

import io # 输入/输出模块
import os # 操作系统接口
import cv2 # OpenCV 包
import numpy as np # 线性代数
import pandas as pd # 数据处理、CSV 文件读写（例如 pd.read_csv）

from urllib import request # 用于打开 HTTP 请求的模块
from matplotlib import pyplot as plt # 绘图库


<div style="width:100%; height:140px">
    <img src="https://www.kuleuven.be/internationaal/thinktank/fotos-en-logos/ku-leuven-logo.png/image_preview" width = auto, heigh = 200px align=left>
</div>

<div class="alert alert-block alert-secondary"><small><b>关于本文件：</b>本笔记本为 <code>ga1_group_X.ipynb</code> 的中文直译版；代码逻辑与原版一致，注释与说明文字已译为中文。</small></div>

KUL H02A5a 计算机视觉：小组作业 1
---------------------------------------------------------------
<div style="height:50px"></div>

<span style="color:red">**待办**：在下方填写你们的姓名。</span>

<span style="color:red">**待办**：将笔记本文件名中的 X 改为你们实际的小组编号。</span>

学生姓名：<span style="color:red">姓名1, 姓名2, ...</span>。

本作业的目标是探索更先进的技术，以构建能更好描述感兴趣物体的特征，并利用这些特征进行人脸识别。本作业以 5 人小组形式提交（小组可由你们自行组成，或由助教随机分配）。

---------------------------------------------------------------
本笔记本结构如下：

0. 数据加载与预处理
1. 特征表示
2. 评价指标
3. 分类器
4. 实验
5. 发布最佳结果
6. 讨论

请确保笔记本**自成体系**且**文档完整**。为你们的设计选择提供充分理由，并说明从实验中得到的洞见。如有疑问，请使用 Toledo 上 *小组作业 1* 的讨论区/论坛。

祝顺利！


<div class="alert alert-block alert-info">
<b>说明：</b>本笔记本仅为示例/模板，可按需任意调整！请保持条理清晰并做好相应文档记录！
</div>

<div class="alert alert-block alert-info">
<b>说明：</b>请清楚标明你们所做的改进！！！例如可使用如下标题：<i>3.1. 改进：带 RBF 核的非线性 SVM。</i>
</div>
    
---------------------------------------------------------------
# 0. 数据加载与预处理

## 0.1. 加载数据
训练集比测试集小很多，这可能看起来有些奇怪；然而这接近真实场景——你们的系统可能会在日常使用中持续运行！在本实验中，我们将尽力利用手头数据做到最好！ 


In [ ]:
# 输入数据文件位于只读的 "../input/" 目录中

train = pd.read_csv(
    '/kaggle/input/kul-computer-vision-ga-1-2025/train_set.csv', index_col = 0)
train.index = train.index.rename('id')

test = pd.read_csv(
    '/kaggle/input/kul-computer-vision-ga-1-2025/test_set.csv', index_col = 0)
test.index = test.index.rename('id')

# 将图像读取为 numpy 数组并存入 "img" 列
train['img'] = [cv2.cvtColor(np.load('/kaggle/input/kul-computer-vision-ga-1-2025/train/train_{}.npy'.format(index), allow_pickle=False), cv2.COLOR_BGR2RGB) 
                for index, row in train.iterrows()]

test['img'] = [cv2.cvtColor(np.load('/kaggle/input/kul-computer-vision-ga-1-2025/test/test_{}.npy'.format(index), allow_pickle=False), cv2.COLOR_BGR2RGB) 
                for index, row in test.iterrows()]
  

train_size, test_size = len(train),len(test)

"训练集包含 {} 个样本，测试集包含 {} 个样本。".format(train_size, test_size)


*说明：本数据集是* [*VGG Face 数据集*](https://www.robots.ox.ac.uk/~vgg/data/vgg_face/) *的一个子集。*

## 0.2. 初览
我们来看一下数据列与类别分布。


In [ ]:
# 训练集包含标识符、姓名、图像信息与类别标签
train.head(1)


In [ ]:
# 测试集仅包含标识符及对应图像信息。

test.head(1)


In [ ]:
# 训练集中的类别分布：
train.groupby('name').agg({'img':'count', 'class': 'max'})


请注意，**Jesse 被赋予分类标签 1**，**Mila 被赋予分类标签 2**。数据集还包含 20 张**长相相似者（分类标签为 0）**的图像以及原始图像。

## 0.3. 预处理数据
### 0.3.1 示例：HAAR 人脸检测器
本示例使用基于 [HAAR 特征的级联分类器](https://opencv-python-tutroals.readthedocs.io/en/latest/py_tutorials/py_objdetect/py_face_detection/py_face_detection.html) 检测人脸，然后将人脸缩放到统一尺寸。若一张图中有多张人脸，仅取第一张。

<div class="alert alert-block alert-info"> <b>说明：</b>可将临时文件写入 <code>/kaggle/temp/</code> 或 <code>../../tmp</code>，但这些文件在当前会话之外不会被保存
</div>



In [ ]:
class HAARPreprocessor():
    """围绕基于 HAAR 特征的级联分类器构建的预处理流水线。 """
    
    def __init__(self, path, face_size):
        self.face_size = face_size
        file_path = os.path.join(path, "haarcascade_frontalface_default.xml")
        if not os.path.exists(file_path): 
            if not os.path.exists(path):
                os.mkdir(path)
            self.download_model(file_path)
        
        self.classifier = cv2.CascadeClassifier(file_path)
  
    def download_model(self, path):
        url = "https://raw.githubusercontent.com/opencv/opencv/master/data/"\
            "haarcascades/haarcascade_frontalface_default.xml"
        
        with request.urlopen(url) as r, open(path, 'wb') as f:
            f.write(r.read())
            
    def detect_faces(self, img):
        """检测图像中的所有人脸。"""
        
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return self.classifier.detectMultiScale(
            img_gray,
            scaleFactor=1.2,
            minNeighbors=5,
            minSize=(30, 30),
            flags=cv2.CASCADE_SCALE_IMAGE
        )
        
    def extract_faces(self, img):
        """返回图像中所有人脸（裁剪后）。"""
        
        faces = self.detect_faces(img)

        return [img[y:y+h, x:x+w] for (x, y, w, h) in faces]
    
    def preprocess(self, data_row):
        faces = self.extract_faces(data_row['img'])
        
        # 若未检测到人脸，返回 NaN 填充图像
        if len(faces) == 0:
            nan_img = np.empty(self.face_size + (3,))
            nan_img[:] = np.nan
            return nan_img
        
        # 仅返回第一张人脸
        return cv2.resize(faces[0], self.face_size, interpolation = cv2.INTER_AREA)
            
    def __call__(self, data):
        return np.stack([self.preprocess(row) for _, row in data.iterrows()]).astype(int)


**可视化**

我们绘制若干示例。


In [ ]:
# 可调整的参数
FACE_SIZE = (100, 100)

def plot_image_sequence(data, n, imgs_per_row=7):
    n_rows = 1 + int(n/(imgs_per_row+1))
    n_cols = min(imgs_per_row, n)

    f,ax = plt.subplots(n_rows,n_cols, figsize=(10*n_cols,10*n_rows))
    for i in range(n):
        if n == 1:
            ax.imshow(data[i])
        elif n_rows > 1:
            ax[int(i/imgs_per_row),int(i%imgs_per_row)].imshow(data[i])
        else:
            ax[int(i%n)].imshow(data[i])
    plt.show()

    
# 预处理后的数据
preprocessor = HAARPreprocessor(path = '../../tmp', face_size=FACE_SIZE)

train_X, train_y = preprocessor(train), train['class'].values
test_X = preprocessor(test)




In [ ]:
# 绘制 Michael 与 Sarah 的人脸

plot_image_sequence(train_X[train_y == 0], n=20, imgs_per_row=10)


In [ ]:
# 绘制 Jesse 的人脸

plot_image_sequence(train_X[train_y == 1], n=30, imgs_per_row=10)


In [ ]:
# 绘制 Mila 的人脸

plot_image_sequence(train_X[train_y == 2], n=30, imgs_per_row=10)


## 0.4. 保存预处理数据（可选）
<div class="alert alert-block alert-info">
<b>说明：</b>可向当前目录（/kaggle/working/）写入最多 20GB；在使用「Save & Run All」创建版本时，这些内容会作为输出保留。可用于保存中间结果。
</div>


In [ ]:
# 保存预处理数据
# prep_path = '/kaggle/working/prepped_data/'
# if not os.path.exists(prep_path):
#     os.mkdir(prep_path)
    
# np.save(os.path.join(prep_path, 'train_X.npy'), train_X)
# np.save(os.path.join(prep_path, 'train_y.npy'), train_y)
# np.save(os.path.join(prep_path, 'test_X.npy'), test_X)

# 加载预处理数据
# prep_path = '/kaggle/working/prepped_data/'
# if not os.path.exists(prep_path):
#     os.mkdir(prep_path)
# train_X = np.load(os.path.join(prep_path, 'train_X.npy'))
# train_y = np.load(os.path.join(prep_path, 'train_y.npy'))
# test_X = np.load(os.path.join(prep_path, 'test_X.npy'))


可以开始正式实验了！


# 1. 特征表示
## 1.0. 示例：恒等特征提取器
示例特征提取器实际上不做任何变换……它只是原样返回输入：
$$
\forall x : f(x) = x.
$$

作为占位符与基类倒是挺合适 ;)。


In [ ]:
class IdentityFeatureExtractor:
    """一个简单的函数，直接返回输入。"""
    
    def transform(self, X):
        return X
    
    def __call__(self, X):
        return self.transform(X)


## 1.1. 基线 1：HOG 特征提取器 / 尺度不变特征变换（SIFT）
...


In [ ]:
class HOGFeatureExtractor(IdentityFeatureExtractor):
    """待办：该特征提取器尚在搭建中"""
    
    def __init__(**params):
        self.params = params
        
    def transform(self, X):
        raise NotImplmentedError


### 1.1.1. t-SNE 图
...


### 1.1.2. 讨论
...


## 1.2. 基线 2：PCA 特征提取器
...


In [ ]:
class PCAFeatureExtractor(IdentityFeatureExtractor):
    """待办：该特征提取器尚在搭建中"""
    
    def __init__(self, n_components):
        self.n_components = n_components
        
    def transform(self, X):
        raise NotImplmentedError
        
    def inverse_transform(self, X):
        raise NotImplmentedError


### 1.2.1. 特征脸（Eigenface）图
...


### 1.2.2. 特征空间图
...


### 1.2.3. 讨论
...


# 2. 评价指标
## 2.0. 示例：准确率
作为示例指标，我们采用准确率。非严格地说，准确率是正确预测数占总预测数的比例。它在分类任务中很常见，但当然也有其缺点……


In [ ]:
from sklearn.metrics import accuracy_score

# 3. 分类器
## 3.0. 示例：*「不太聪明」*的分类器
该随机分类器并不复杂。它根据训练集中观察到的类别分布进行随机预测。**因此它假设**测试集的类别标签分布与训练集相似。


In [ ]:
class RandomClassificationModel:
    """随机分类器：根据训练阶段观察到的类别分布进行随机抽样。"""
    
    def fit(self, X, y):
        """将类别比例调整为 y 中观察到的分布。

        参数
        ----------
        X : tensor
            训练集
        y : array
            训练集标签

        返回
        -------
        self : RandomClassificationModel
        """
        
        self.classes, self.class_ratio = np.unique(y, return_counts=True)
        self.class_ratio = self.class_ratio / self.class_ratio.sum()
        return self
        
    def predict(self, X):
        """为输入数据抽样标签。

        参数
        ----------
        X : tensor
            数据集
            
        返回
        -------
        y_star : array
            「预测」标签
        """

        np.random.seed(0)
        return np.random.choice(self.classes, size = X.shape[0], p=self.class_ratio)
    
    def __call__(self, X):
        return self.predict(X)
    


## 3.1. 基线 1：我最喜欢的分类器
...


In [ ]:
class FavoriteClassificationModel:
    """待办：该分类器尚在搭建中。"""
    
    def fit(self, X, y):
        raise NotImplmentedError
        
    def predict(self, X):
        raise NotImplmentedError


# 4. 实验
<div class="alert alert-block alert-info"> <b>说明：</b><i>不要</i>用本节记录代码中的每一处微小改动！请突出最重要的发现以及你们找到的主要（最佳）流水线。
</div>
<br>

## 4.0. 示例：基础流水线
基础流水线对任意输入按训练集的类别标签分布抽样标签。如预期，性能很差，在训练集上大约只有约 1/4 预测正确。改进空间很大，留给你们完成 ;)。 


In [ ]:
feature_extractor = IdentityFeatureExtractor() 
classifier = RandomClassificationModel()

# 在特征上训练模型
classifier.fit(feature_extractor(train_X), train_y)

# 模型 / 最终流水线
model = lambda X: classifier(feature_extractor(X))


In [ ]:
# 在训练集上评估模型性能
train_y_star = model(train_X)

"在训练集上的性能为 {:.2f}。然而，这并不能很好反映真实性能（泛化能力）。".format(
    accuracy_score(train_y, train_y_star))


In [ ]:
# 预测测试集标签
test_y_star = model(test_X)


# 5. 发布最佳结果


In [ ]:
submission = test.copy().drop('img', axis = 1)
submission['class'] = test_y_star

submission

In [ ]:
submission.to_csv('submission.csv')

# 6. 讨论
...

总结而言，我们贡献如下：
* 

